In [1]:
import pandas as pd
import numpy as np

In [2]:
oos_independent = pd.read_csv("oos_independent.csv")

oos_independent.head()

,idx,date,id,league_name,has_odds,result,best_A,best_D,best_H,model,uses_odds_as_feature,pred,prob_A,prob_D,prob_H,max_prob,correct
0,6500,2010-05-11,16131,Poland Ekstraklasa,False,A,NaN,NaN,NaN,logreg_l2,False,D,0.199763,0.460142,0.340095,0.460142,False
1,6500,2010-05-11,16131,Poland Ekstraklasa,False,A,NaN,NaN,NaN,decision_tree,False,H,0.253482,0.318477,0.428041,0.428041,False
2,6500,2010-05-11,16131,Poland Ekstraklasa,False,A,NaN,NaN,NaN,random_forest,False,H,0.239563,0.287750,0.472688,0.472688,False
3,6500,2010-05-11,16131,Poland Ekstraklasa,False,A,NaN,NaN,NaN,hist_gradient_boosting,False,H,0.277894,0.289724,0.432382,0.432382,False
4,6501,2010-05-11,16132,Poland Ekstraklasa,False,H,NaN,NaN,NaN,logreg_l2,False,H,0.181582,0.196236,0.622182,0.622182,True


In [3]:
oos_independent.columns.tolist()

['idx',
 'date',
 'id',
 'league_name',
 'has_odds',
 'result',
 'best_A',
 'best_D',
 'best_H',
 'model',
 'uses_odds_as_feature',
 'pred',
 'prob_A',
 'prob_D',
 'prob_H',
 'max_prob',
 'correct']

In [4]:
def calculate_ev(prob, odds):
    return (prob * odds) - 1

In [5]:
def fractional_kelly(prob, odds, kelly_fraction=0.25):

    b = odds - 1

    kelly = ((prob * odds) - 1) / b

    return max(0, kelly * kelly_fraction)

In [6]:
BETTING_STRATEGIES = {

    "conservative": {
        "ev_threshold": 0.10,
        "kelly_fraction": 0.10,
        "max_bet_frac": 0.01,
        "min_confidence": 0.60,
        "min_odds": 1.50,
        "max_odds": 4.00
    },

    "balanced": {
        "ev_threshold": 0.05,
        "kelly_fraction": 0.25,
        "max_bet_frac": 0.02,
        "min_confidence": 0.50,
        "min_odds": 1.30,
        "max_odds": 6.00
    },

    "aggressive": {
        "ev_threshold": 0.02,
        "kelly_fraction": 0.50,
        "max_bet_frac": 0.05,
        "min_confidence": 0.40,
        "min_odds": 1.01,
        "max_odds": 15.00
    }
}

In [7]:
def simulate_betting_strategy(
    df,
    strategy_name,
    strategy_params,
    starting_bankroll=10000
):

    bankroll = starting_bankroll

    bets = []

    outcomes = ["H", "D", "A"]

    for _, row in df.iterrows():

        best_bet = None
        best_ev = -999

        for outcome in outcomes:

            prob = row[f"prob_{outcome}"]
            odds = row[f"best_{outcome}"]

            if pd.isna(prob) or pd.isna(odds):
                continue

            ev = calculate_ev(prob, odds)

            if ev > best_ev:
                best_ev = ev
                best_bet = outcome

        if best_bet is None:
            continue

        prob = row[f"prob_{best_bet}"]
        odds = row[f"best_{best_bet}"]

        confidence = prob

        # filters
        if best_ev < strategy_params["ev_threshold"]:
            continue

        if confidence < strategy_params["min_confidence"]:
            continue

        if odds < strategy_params["min_odds"]:
            continue

        if odds > strategy_params["max_odds"]:
            continue

        # Kelly stake
        stake_frac = fractional_kelly(
            prob,
            odds,
            strategy_params["kelly_fraction"]
        )

        stake_frac = min(
            stake_frac,
            strategy_params["max_bet_frac"]
        )

        stake = bankroll * stake_frac

        if stake <= 0:
            continue

        won = row["result"] == best_bet

        if won:
            profit = stake * (odds - 1)
        else:
            profit = -stake

        bankroll += profit

        bets.append({
            "model": row["model"],
            "strategy": strategy_name,
            "bet_outcome": best_bet,
            "ev": best_ev,
            "confidence": confidence,
            "odds": odds,
            "stake": stake,
            "won": won,
            "profit": profit,
            "bankroll": bankroll
        })

    bets_df = pd.DataFrame(bets)

    if len(bets_df) == 0:
        return None

    total_profit = bets_df["profit"].sum()

    total_staked = bets_df["stake"].sum()

    roi = total_profit / total_staked

    hit_rate = bets_df["won"].mean()

    return {
        "strategy": strategy_name,
        "bets": len(bets_df),
        "hit_rate": hit_rate,
        "profit": total_profit,
        "roi": roi,
        "end_bankroll": bankroll,
        "total_staked": total_staked
    }

In [8]:
results = []

for model_name, df_model in oos_independent.groupby("model"):

    for strategy_name, strategy_params in BETTING_STRATEGIES.items():

        result = simulate_betting_strategy(
            df=df_model,
            strategy_name=strategy_name,
            strategy_params=strategy_params
        )

        if result is not None:

            result["model"] = model_name

            results.append(result)

In [9]:
betting_results = pd.DataFrame(results)

betting_results = betting_results.sort_values(
    "roi",
    ascending=False
)

betting_results.round(4)

,strategy,bets,hit_rate,profit,roi,end_bankroll,total_staked,model
1,aggressive,36,0.5278,263.6810,0.0568,10263.6810,4643.0704,bookmaker
6,balanced,2660,0.5376,6839.4289,0.0132,16839.4289,516289.5000,hist_gradient_boosting
3,balanced,3611,0.5289,-332.6691,-0.0009,9667.3309,359810.1545,decision_tree
2,conservative,1456,0.5412,-841.6693,-0.0077,9158.3307,109096.5667,decision_tree
8,conservative,1304,0.5337,-2329.6762,-0.0191,7670.3238,122194.6460,logreg_l2
12,balanced,1563,0.5195,-4535.1643,-0.0257,5464.8357,176779.9299,random_forest
11,conservative,377,0.5438,-1163.3701,-0.0329,8836.6299,35384.4454,random_forest
9,balanced,3667,0.5097,-8232.1612,-0.0345,1767.8388,238408.7474,logreg_l2
5,conservative,763,0.5347,-3219.9508,-0.0526,6780.0492,61211.8162,hist_gradient_boosting
10,aggressive,6965,0.4696,-9999.9438,-0.0599,0.0562,166852.3422,logreg_l2
